In [1]:
import base64
import html
import json
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parents[1]))

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from bernina_prep.diffraction import (
    build_ubcalc,
    k_magnitude,
    make_constraints,
    reciprocal_basis,
    solution_angles,
    solve_hkl,
    surface_normal_phi,
)

# Narrower value fields than before, to leave room for the per-angle solution
# filters next to them; filters are half that width again.
UC_W = "150px"
FIELD_W = "98px"
FILTER_W = "49px"
HKL_W = "98px"
DESC_W = "58px"
LABEL_W = "82px"

LAT_NAME = "custom lattice"

# This applet speaks SwissFEL lab axes throughout: x horizontal transverse,
# y vertical, z along the beam. diffcalc uses the same right-handed frame with
# a cyclic relabelling (diffcalc +x,+y,+z = SwissFEL +y,+z,+x), and
# bernina_prep.diffraction converts at its boundary.
LAB_FRAME = "swissfel"

# One standard ordering, used for the inputs AND the solution rows: the six
# physical circles first, then diffcalc's virtual/derived angles.
PHYSICAL_ANGLES = ["gam", "delta", "mu", "eta", "chi", "phi"]
VIRTUAL_ANGLES = ["qaz", "naz", "alpha", "beta", "psi", "a_eq_b", "mu_is_gam"]
angle_names = PHYSICAL_ANGLES + VIRTUAL_ANGLES
BOOL_ANGLES = {"a_eq_b", "mu_is_gam"}

# solve_hkl reports 'gam' as 'gamma'; same order, plus the derived quantities.
ROW_ORDER = ["gamma", "delta", "mu", "eta", "chi", "phi",
             "theta", "ttheta", "qaz", "naz", "alpha", "beta", "psi",
             "tau", "betain", "betaout"]

default_fixed = {"eta", "chi", "phi"}

ctrl = {
    "uc_a": widgets.FloatText(description="a (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_b": widgets.FloatText(description="b (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_c": widgets.FloatText(description="c (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_alpha": widgets.FloatText(description="alpha", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_beta": widgets.FloatText(description="beta", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_gamma": widgets.FloatText(description="gamma", value=90, layout=widgets.Layout(width=UC_W)),
    "h": widgets.IntText(value=1, description="h", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": "20px"}),
    "k": widgets.IntText(value=1, description="k", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": "20px"}),
    "l": widgets.IntText(value=1, description="l", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": "20px"}),
    # Lab-frame directions are entered in SwissFEL axes (x transverse, y up,
    # z beam) and converted to diffcalc's on the way in - see LAB_FRAME below.
    # The surface normal is an (h,k,l), so it carries no lab frame at all.
    "surface_normal": widgets.Text(value="(1,1,1)",
                                   description="surf. normal (h,k,l)",
                                   layout=widgets.Layout(width="372px"),
                                   style={"description_width": "168px"}),
    "orientations": widgets.Textarea(
        value="(1,1,1) > (1,0,0)\n(1,1,-1) > (0,1,0)",
        description="orientations (h,k,l) > (x,y,z)",
        placeholder='one "(h,k,l) > (x,y,z)" pair per line, xyz in SwissFEL axes',
        layout=widgets.Layout(width="372px", height="66px"),
        style={"description_width": "168px"},
    ),
}

# per-angle solution filters; an empty box means "no constraint"
filters = {}
angle_rows = []
for name in angle_names:
    if name in BOOL_ANGLES:
        ctrl[name] = widgets.Checkbox(value=False, indent=False,
                                      layout=widgets.Layout(width=FIELD_W))
    else:
        ctrl[name] = widgets.FloatText(value=0, layout=widgets.Layout(width=FIELD_W))
    ctrl[f"{name}_fixed"] = widgets.Checkbox(value=name in default_fixed,
                                             description="fix", indent=False,
                                             layout=widgets.Layout(width="62px"))
    row = [widgets.Label(name, layout=widgets.Layout(width=LABEL_W)), ctrl[name]]
    if name in BOOL_ANGLES:
        row.append(widgets.Label("", layout=widgets.Layout(width="106px")))
    else:
        lo = widgets.Text(placeholder="min", layout=widgets.Layout(width=FILTER_W))
        hi = widgets.Text(placeholder="max", layout=widgets.Layout(width=FILTER_W))
        filters[name] = (lo, hi)
        row += [lo, hi]
    row.append(ctrl[f"{name}_fixed"])
    angle_rows.append(widgets.HBox(row))

ctrl["energy"] = widgets.FloatSlider(value=8000, min=1000, max=20000, step=10,
                                     readout=False, continuous_update=False,
                                     layout=widgets.Layout(width="170px"))
energy_value = widgets.BoundedFloatText(description="energy", value=8000, min=1000,
                                        max=20000, step=10,
                                        layout=widgets.Layout(width="150px"),
                                        style={"description_width": DESC_W})
widgets.link((energy_value, "value"), (ctrl["energy"], "value"))
energy_row = widgets.HBox([energy_value, widgets.Label("eV"), ctrl["energy"]])

ctr_uc = widgets.HBox([ctrl[n] for n in ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma")],
                      layout=widgets.Layout(flex_flow="row wrap"))
ctr_hkl = widgets.HBox([ctrl["h"], ctrl["k"], ctrl["l"]],
                       layout=widgets.Layout(flex_flow="row wrap"))
ctr_or = widgets.VBox([ctrl["surface_normal"], ctrl["orientations"]])
filter_header = widgets.HTML(
    f'<div style="font-size:11px;color:#666;margin-top:6px">'
    f'<span style="display:inline-block;width:{LABEL_W}"></span>'
    f'<span style="display:inline-block;width:{FIELD_W}">value</span>'
    f'<span style="display:inline-block;width:106px">solution filter</span>'
    f'</div>')
ctr_ang = widgets.VBox([filter_header] + angle_rows)

# --- stored beamline presets ------------------------------------------------
# JSON files in apps/bernina_parameters/. These deliberately apply ONLY the
# angle values, the fix conditions and the solution filters - whatever sample
# and hkl you have entered is left untouched - and then solve once.

PRESET_DIR = Path.cwd().parent / "bernina_parameters"
PRESET_GROUPS = ("angles", "fixed", "filters")

# While a whole config is being applied, every widget write would otherwise
# fire its own solve; suspend and run exactly one at the end.
_suspend = {"on": False}


def preset_label(path):
    """Bernina_THC_diffraction_config.json -> 'Bernina THC'."""
    stem = path.stem
    for suffix in ("_diffraction_config", "_config"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break
    return stem.replace("_", " ").strip()


def find_presets():
    if not PRESET_DIR.is_dir():
        return {}
    return {preset_label(p): p for p in sorted(PRESET_DIR.glob("*.json"))}


presets = find_presets()
PRESET_NONE = "— preset —"
preset_dd = widgets.Dropdown(
    options=[PRESET_NONE] + list(presets),
    value=PRESET_NONE,
    description="load preset",
    layout=widgets.Layout(width="330px"),
    style={"description_width": "90px"},
)
preset_msg = widgets.HTML()


def on_preset(change):
    name = change["new"]
    if name == PRESET_NONE or name not in presets:
        return
    try:
        cfg = json.loads(presets[name].read_text())
    except Exception as exc:
        preset_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        return
    subset = {k: v for k, v in cfg.items() if k in PRESET_GROUPS}
    ignored = [k for k in cfg if k not in PRESET_GROUPS]
    _suspend["on"] = True
    try:
        apply_config(subset)
    finally:
        _suspend["on"] = False
    recompute()                      # exactly one solve after the batch
    note = f"loaded <b>{html.escape(name)}</b>: {', '.join(sorted(subset))}"
    if ignored:
        note += f" (ignored {', '.join(sorted(ignored))}; sample and hkl kept)"
    preset_msg.value = f'<span style="font-size:11px;color:#282">{note}</span>'


preset_dd.observe(on_preset, names="value")


# --- store / recall configurations ------------------------------------------

INCLUDE_GROUPS = [
    ("sample", "sample parameters (unit cell, orientations, surface normal)"),
    ("hkl", "h, k, l"),
    ("energy", "energy"),
    ("angles", "angle values"),
    ("fixed", "fix conditions"),
    ("filters", "solution filters"),
]
include_boxes = {
    key: widgets.Checkbox(value=True, description=label, indent=False,
                          layout=widgets.Layout(width="330px"))
    for key, label in INCLUDE_GROUPS
}

store_btn = widgets.Button(description="store config…", layout=widgets.Layout(width="130px"))
make_btn = widgets.Button(description="create file", button_style="primary",
                          layout=widgets.Layout(width="110px"))
upload = widgets.FileUpload(accept=".json", multiple=False,
                            description="load config", layout=widgets.Layout(width="150px"))
download_link = widgets.HTML()
config_msg = widgets.HTML()
store_panel = widgets.VBox(
    [widgets.HTML('<div style="font-size:11px;color:#666">include in the file '
                  '(unchecked groups are left out, so the file acts as a partial '
                  'template and anything missing stays free on load):</div>')]
    + [include_boxes[k] for k, _ in INCLUDE_GROUPS]
    + [widgets.HBox([make_btn]), download_link],
    layout=widgets.Layout(display="none", border="1px solid #ddd",
                          padding="6px", margin="4px 0 0 0"),
)


def toggle_store(_b):
    shown = store_panel.layout.display != "none"
    store_panel.layout.display = "none" if shown else "flex"


store_btn.on_click(toggle_store)


def current_config():
    cfg = {}
    if include_boxes["sample"].value:
        cfg["sample"] = {n: ctrl[n].value for n in
                         ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma",
                          "surface_normal", "orientations")}
    if include_boxes["hkl"].value:
        cfg["hkl"] = {n: ctrl[n].value for n in ("h", "k", "l")}
    if include_boxes["energy"].value:
        cfg["energy"] = ctrl["energy"].value
    if include_boxes["angles"].value:
        cfg["angles"] = {n: ctrl[n].value for n in angle_names}
    if include_boxes["fixed"].value:
        cfg["fixed"] = {n: ctrl[f"{n}_fixed"].value for n in angle_names}
    if include_boxes["filters"].value:
        # kept as strings, empty included on purpose: an empty filter is a
        # meaningful "free" state in a template
        cfg["filters"] = {n: [lo.value, hi.value] for n, (lo, hi) in filters.items()}
    return cfg


def on_make(_b):
    cfg = current_config()
    if not cfg:
        download_link.value = '<span style="color:#b00">nothing selected</span>'
        return
    blob = base64.b64encode(json.dumps(cfg, indent=2).encode()).decode()
    download_link.value = (
        f'<a download="bernina_diffraction_config.json" '
        f'href="data:application/json;base64,{blob}" '
        f'style="font-size:12px">&#11015; download bernina_diffraction_config.json</a>')


make_btn.on_click(on_make)


def apply_config(cfg):
    applied = []
    for name, value in (cfg.get("sample") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("hkl") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    if "energy" in cfg:
        ctrl["energy"].value = cfg["energy"]
        applied.append("energy")
    for name, value in (cfg.get("angles") or {}).items():
        if name in ctrl:
            ctrl[name].value = value
            applied.append(name)
    for name, value in (cfg.get("fixed") or {}).items():
        if f"{name}_fixed" in ctrl:
            ctrl[f"{name}_fixed"].value = bool(value)
            applied.append(f"{name}_fixed")
    for name, pair in (cfg.get("filters") or {}).items():
        if name in filters:
            filters[name][0].value = pair[0] or ""
            filters[name][1].value = pair[1] or ""
            applied.append(f"{name}_filter")
    return applied


def on_upload(change):
    items = change["new"]
    if not items:
        return
    item = items[0] if isinstance(items, (list, tuple)) else list(items.values())[0]
    try:
        raw = item["content"]
        cfg = json.loads(bytes(raw).decode())
        _suspend["on"] = True
        try:
            applied = apply_config(cfg)
        finally:
            _suspend["on"] = False
        recompute()
        config_msg.value = (f'<span style="font-size:11px;color:#282">loaded '
                            f'{len(applied)} settings; groups absent from the file '
                            f'were left as they are</span>')
    except Exception as exc:
        config_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'


upload.observe(on_upload, names="value")

config_row = widgets.VBox([preset_dd, preset_msg,
                           widgets.HBox([store_btn, upload]), store_panel, config_msg])

controls = widgets.VBox([ctr_uc, ctr_hkl, ctr_or, ctr_ang, energy_row,
                         widgets.HTML('<hr style="margin:8px 0">'), config_row],
                        layout=widgets.Layout(width="380px", flex="0 0 auto"))

# --- solutions table, one clickable column per (surviving) branch ------------

ROW_LABEL_W = 78
COL_W = 96
SELECTED_BG = "#cfe8fb"

state = {"df": None, "cols": [], "selected": None, "ubcalc": None, "cons": None}

solution_buttons = widgets.HBox([])
table_html = widgets.HTML()
solve_msg = widgets.HTML()
output = widgets.Output()


def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NaN"
    return f"{float(v):.3f}"


def _num_or_none(text):
    text = (text or "").strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def passing_columns(df):
    """Columns surviving the per-angle min/max filters (empty box = ignored)."""
    keep = []
    for col in df.columns:
        ok = True
        for name, (lo_w, hi_w) in filters.items():
            row = "gamma" if name == "gam" else name
            if row not in df.index:
                continue
            lo, hi = _num_or_none(lo_w.value), _num_or_none(hi_w.value)
            if lo is None and hi is None:
                continue
            v = df.loc[row, col]
            if pd.isna(v) or (lo is not None and v < lo) or (hi is not None and v > hi):
                ok = False
                break
        if ok:
            keep.append(col)
    return keep


def render_table():
    df, cols = state["df"], state["cols"]
    if df is None or not cols:
        table_html.value = ("" if df is None else
                            '<span style="font-size:12px;color:#a60">'
                            'no solution passes the filters</span>')
        return
    sel = state["selected"]
    head = "".join(
        f'<th style="width:{COL_W}px;text-align:right;padding:2px 6px;'
        f'border-bottom:1px solid #ccc;'
        f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">{html.escape(str(c))}</th>'
        for i, c in enumerate(cols))
    body = []
    for name in df.index:
        cells = "".join(
            f'<td style="width:{COL_W}px;text-align:right;padding:1px 6px;'
            f'font-variant-numeric:tabular-nums;'
            f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">'
            f'{_fmt(df.loc[name, c])}</td>'
            for i, c in enumerate(cols))
        body.append(
            f'<tr><td style="width:{ROW_LABEL_W}px;text-align:right;padding:1px 6px;'
            f'font-weight:600;">{html.escape(str(name))}</td>{cells}</tr>')
    table_html.value = ('<table style="border-collapse:collapse;font-size:12px;">'
                        f'<tr><td style="width:{ROW_LABEL_W}px"></td>{head}</tr>'
                        + "".join(body) + "</table>")


def select_solution(index):
    state["selected"] = index
    for i, b in enumerate(solution_buttons.children[1:]):
        b.button_style = "info" if i == index else ""
    if solution_buttons.children:
        solution_buttons.children[0].button_style = "" if index is not None else "info"
    render_table()
    push_view()


def rebuild_solution_buttons():
    buttons = []
    for i, col in enumerate(state["cols"]):
        b = widgets.Button(description=str(col), tooltip=f"show {col} in the 3D view",
                           layout=widgets.Layout(width=f"{COL_W}px", margin="0"))
        b.on_click(lambda _b, idx=i: select_solution(idx))
        buttons.append(b)
    zero = widgets.Button(description="zero", tooltip="move all circles back to 0",
                          layout=widgets.Layout(width=f"{ROW_LABEL_W}px", margin="0"))
    zero.on_click(lambda _b: select_solution(None))
    solution_buttons.children = [zero] + buttons


# --- the 3D view ------------------------------------------------------------

_html_path = Path.cwd() / "diffractometer_3d.html"
_scene_html = _html_path.read_text()

view_frame = widgets.HTML(
    f'<iframe id="diff3d" srcdoc="{html.escape(_scene_html)}" '
    'style="width:100%; height:760px; border:1px solid #ddd;"></iframe>',
    layout=widgets.Layout(width="100%"),
)
payload_holder = widgets.HTML()
_push_count = {"n": 0}


def push_view():
    angles = {n: 0.0 for n in ("mu", "eta", "chi", "phi", "delta", "gamma")}
    label = ""
    if state["df"] is not None and state["selected"] is not None and state["cols"]:
        col = state["cols"][state["selected"]]
        angles = solution_angles(state["df"], col)
        label = str(col)
    msg = {"angles": angles, "kMag": k_magnitude(ctrl["energy"].value),
           "label": label, "n": _push_count["n"]}
    if state["ubcalc"] is not None:
        # the view derives the measured hkl itself, live, from UB + the angles
        msg["ub"] = reciprocal_basis(state["ubcalc"])
        # so the sample platelet shows the surface orientation actually entered
        msg["nPhi"] = surface_normal_phi(state["ubcalc"])
    _push_count["n"] += 1
    payload_holder.value = ('<span id="diff3d-payload" style="display:none">'
                            + html.escape(json.dumps(msg)) + "</span>")


# --- solve ------------------------------------------------------------------

def recompute(*_):
    global last_ubcalc, last_cons
    if _suspend["on"]:
        return
    try:
        ubcalc = build_ubcalc(
            LAT_NAME,
            ctrl["uc_a"].value, ctrl["uc_b"].value, ctrl["uc_c"].value,
            ctrl["uc_alpha"].value, ctrl["uc_beta"].value, ctrl["uc_gamma"].value,
            [ln.strip() for ln in ctrl["orientations"].value.splitlines() if ln.strip()],
            ctrl["surface_normal"].value,
            frame=LAB_FRAME,
        )
        fixed = {n: ctrl[n].value for n in angle_names if ctrl[f"{n}_fixed"].value}
        cons = make_constraints(fixed)
        df = solve_hkl(ubcalc, cons, ctrl["h"].value, ctrl["k"].value,
                       ctrl["l"].value, ctrl["energy"].value)
    except Exception as exc:
        solve_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        state.update(df=None, cols=[])
        rebuild_solution_buttons()
        render_table()
        push_view()
        return

    # one standard row order, matching the input ordering
    ordered = [r for r in ROW_ORDER if r in df.index] + \
              [r for r in df.index if r not in ROW_ORDER]
    df = df.reindex(ordered)

    solve_msg.value = ""
    cols = passing_columns(df)
    state.update(df=df, cols=cols, ubcalc=ubcalc, cons=cons)
    last_ubcalc, last_cons = ubcalc, cons
    if state["selected"] is not None and state["selected"] >= len(cols):
        state["selected"] = None
    rebuild_solution_buttons()
    select_solution(state["selected"])


for _w in ctrl.values():
    _w.observe(recompute, names="value")
for _lo, _hi in filters.values():
    _lo.observe(recompute, names="value")
    _hi.observe(recompute, names="value")
recompute()

solutions_panel = widgets.VBox(
    [widgets.HTML("<b>solutions</b> &mdash; click a column to show it in the 3D view"),
     solve_msg, solution_buttons, table_html],
    layout=widgets.Layout(flex="0 0 auto", margin="0 0 0 12px"),
)
view_panel = widgets.VBox(
    [view_frame, payload_holder],
    layout=widgets.Layout(flex="1 1 520px", min_width="0", margin="0 0 0 12px"),
)

display(widgets.HBox(
    [controls, solutions_panel, view_panel],
    layout=widgets.Layout(flex_flow="row wrap", width="100%", align_items="flex-start"),
))


In [ ]:

print("PRESET_DIR:", PRESET_DIR, "exists:", PRESET_DIR.is_dir())
print("presets:", list(presets))
print("dropdown options:", preset_dd.options)
print("chi before:", ctrl["chi"].value, "chi_fixed:", ctrl["chi_fixed"].value,
      "delta filter:", filters["delta"][0].value, filters["delta"][1].value)
import traceback
try:
    on_preset({"new": "Bernina THC"})
except Exception:
    traceback.print_exc()
print("chi after :", ctrl["chi"].value, "chi_fixed:", ctrl["chi_fixed"].value,
      "delta filter:", filters["delta"][0].value, filters["delta"][1].value)
print("preset_msg:", preset_msg.value[:200])
print("surviving solution columns:", state["cols"])
